In [200]:
import yaml
import copy
from datetime import datetime,timezone,timedelta

In [201]:
def load_yaml(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)
    
config  = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml") 

In [202]:
op1 = {
    'section': 'Profile',
    'scores': {
        'ContentQuality': {
            'score': 1,
            'feedback': "xxx"},
        'Completeness': {
            'score': 1,
            'feedback': "yyy"}
        },
    'session_feedback': 'zzz'
}

In [203]:
op3 = {
    'section': 'Education',
    'scores': {
        'RoleRelevance': {
            'score': 5,
            'feedback': 'abc'},
        'Completeness': {
        'score': 5,
        'feedback': "def"}
        },
    'session_feedback': 'ghi'
}

In [204]:
def aggregate(llm_output:dict,verbose=0):
    '''
    convert, reshape, transform output format that We got from llm
    '''
    # print(f"llm_output ->\n{llm_output}")
    llm_output       = llm_output                  # op
    section_name     = llm_output["section"]       # Get section
    section_config_scores   = config["weights"][section_name]  # config["weights"][section_key][criteria]
    scaled_criteria_scores = {} 
    total_section_raw_score = 0.0 
    total_section_max_score  = 0.0
    scores_copy = copy.deepcopy(llm_output["scores"])  # Protect multiple mutation when we run more than one time
    for criteria,body in scores_copy.items():
        raw_llm_score = body["score"]             # score ดิบๆ ที่ออกมาจาก LLM
        if raw_llm_score == 0:
            max_score_from_config = 0                  # Now It's 10 (default)
        else:
            max_score_from_config = section_config_scores[criteria]   # max score ที่ set ใน weight.yaml เพื่อ scale เต็มๆ
        scaled_score = (raw_llm_score / 5) * max_score_from_config          # raw / normalize max score * weight scale
        body["score"] = scaled_score
        scaled_criteria_scores[criteria] = body
        total_section_raw_score = total_section_raw_score + scaled_score
        total_section_max_score = total_section_max_score + max_score_from_config
        if verbose:
            print(f"criteria->{criteria}")
            print(f"body->{body}")
            print(f" - raw -> {raw}") 
            print(f" - w   -> {w}")
            print(f" - weighted -> {weighted}")  
            print(f" - body['score'] -> {body['score']}")  
            print(f"total -> {total}")
            print()
    return {
            "section": section_name,
            "total_section_raw_score":total_section_raw_score,
            "total_section_max_score":total_section_max_score,
            "scores":scaled_criteria_scores,
            "session_feedback":llm_output['session_feedback']
        }

In [223]:
1/5 * 10 + 1/5*10
5/5 * 10 + 5/5*10

20.0

In [205]:
print("s1")
s1 = aggregate(op1)
print("s2")
s2 = aggregate(op3)

s1
s2


In [206]:
s1

{'section': 'Profile',
 'total_section_raw_score': 4.0,
 'total_section_max_score': 20.0,
 'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
  'Completeness': {'score': 2.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [207]:
s2

{'section': 'Education',
 'total_section_raw_score': 20.0,
 'total_section_max_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
  'Completeness': {'score': 10.0, 'feedback': 'def'}},
 'session_feedback': 'ghi'}

<hr>

In [208]:
section_outputs = [s1,s2]
timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
model_config    = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
weight_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
prompt_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml") 
config_lang     = prompt_config['Language_output_style']["en"]

In [209]:
section_outputs

[{'section': 'Profile',
  'total_section_raw_score': 4.0,
  'total_section_max_score': 20.0,
  'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
   'Completeness': {'score': 2.0, 'feedback': 'yyy'}},
  'session_feedback': 'zzz'},
 {'section': 'Education',
  'total_section_raw_score': 20.0,
  'total_section_max_score': 20.0,
  'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
   'Completeness': {'score': 10.0, 'feedback': 'def'}},
  'session_feedback': 'ghi'}]

In [ ]:
# # 2 x 0.4 = 0.8
# # 10 x 0.4 = 4

# # 20 x 0.6 = 12
# # 20 x 0.6 = 12

# # ->

# 12 x 0.4 = 4.8
# 20 x 0.4 = 8
# = 4.8/8

# 20 x 0.6 = 12
# 20 x 0.6 = 12
# = 3/3

# 4 * 0.4

25.6

In [220]:
(1/5*10)

2.0

In [225]:
4 * 0.4 + 20 * 0.6

13.6

In [212]:
weights      = weight_config["weights"]
contribution = {}
total_weighted_raw_score  = 0.0
total_weighted_max_score  = 0.0
for section_data in section_outputs:
    print(f"section_data->{section_data}")
    section_name            = section_data["section"]
    total_section_raw_score = section_data["total_section_raw_score"]
    total_section_max_score = section_data["total_section_max_score"]
    section_weight          = weights[section_name]["section_weight"]
    
    total_section_raw_score_x_weight = total_section_raw_score*section_weight
    total_section_max_score_x_weight = total_section_max_score*section_weight

    contribution[section_name] = {
        "section_total_score":total_section_raw_score,
        "section_full_score":total_section_max_score,
        "section_weight":section_weight,
        "total_section_raw_score_x_weight":total_section_raw_score_x_weight,
        "total_section_max_score_x_weight":total_section_max_score_x_weight
    }
    total_weighted_raw_score  = total_weighted_raw_score + total_section_raw_score_x_weight
    total_weighted_max_score  = total_weighted_max_score + total_section_max_score_x_weight
    
returnx = {
    "total_weighted_raw_score":total_weighted_raw_score,
    "total_weighted_max_score":total_weighted_max_score,
    "section_contribution":contribution
}
returnx

section_data->{'section': 'Profile', 'total_section_raw_score': 4.0, 'total_section_max_score': 20.0, 'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'}, 'Completeness': {'score': 2.0, 'feedback': 'yyy'}}, 'session_feedback': 'zzz'}
section_data->{'section': 'Education', 'total_section_raw_score': 20.0, 'total_section_max_score': 20.0, 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'}, 'Completeness': {'score': 10.0, 'feedback': 'def'}}, 'session_feedback': 'ghi'}


{'total_weighted_raw_score': 13.6,
 'total_weighted_max_score': 20.0,
 'section_contribution': {'Profile': {'section_total_score': 4.0,
   'section_full_score': 20.0,
   'section_weight': 0.4,
   'total_section_raw_score_x_weight': 1.6,
   'total_section_max_score_x_weight': 8.0},
  'Education': {'section_total_score': 20.0,
   'section_full_score': 20.0,
   'section_weight': 0.6,
   'total_section_raw_score_x_weight': 12.0,
   'total_section_max_score_x_weight': 12.0}}}

In [226]:
def normalize_score(point_score,full_score):
    print(f"Point_score -> {point_score}")
    print(f"full_score  -> {full_score}")
    normalize_score = (point_score/full_score) * 100
    print(f"normalize_score -> {normalize_score}/100")
    return normalize_score

def score_to_grade(score: float) -> str:
    if 90 <= score <= 100:
        return "A"
    elif 80 <= score < 90:
        return "B"
    elif 70 <= score < 80:
        return "C"
    elif 60 <= score < 70:
        return "D"
    elif 0 <= score < 60:
        return "F"
    else:
        return "Gradding error"

In [227]:
norm_score = normalize_score(returnx["total_weighted_raw_score"],returnx["total_weighted_max_score"])
score_to_grade(norm_score)

Point_score -> 13.6
full_score  -> 20.0
normalize_score -> 68.0/100


'D'